# 01 — VoxCeleb1 Exploration

## Objective

Explore VoxCeleb1 as a candidate dataset for speaker identity and privacy evaluation.

This notebook:
- inventories speakers, videos, and utterances;
- loads and aggregates language predictions;
- selects English videos from speakers with at least three videos;
- creates reproducible train/validation/test splits at the **video level**;
- builds, validates, and saves an utterance-level manifest.

The resulting manifest is saved to `data/metadata/voxceleb_utterances.parquet`.

This notebook orchestrates the workflow and displays diagnostics.

## 1. Setup and paths

In [1]:
import sys
import importlib
from pathlib import Path

import pandas as pd

# This notebook is expected to run from the project's notebooks/ directory.
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Reload project modules so edits in src/ are picked up in this kernel.
import paths
import metadata
import voxceleb
import splits

importlib.reload(paths)
importlib.reload(metadata)
importlib.reload(voxceleb)
importlib.reload(splits)

PROJECT_ROOT = paths.PROJECT_ROOT
VOXCELEB_WAV_DIR = paths.VOXCELEB_WAV_DIR
METADATA_DIR = paths.METADATA_DIR

if not VOXCELEB_WAV_DIR.is_dir():
    raise FileNotFoundError(f"VoxCeleb WAV directory not found: {VOXCELEB_WAV_DIR}")
if not METADATA_DIR.is_dir():
    raise FileNotFoundError(f"Metadata directory not found: {METADATA_DIR}")

print("Project root:", PROJECT_ROOT)
print("VoxCeleb WAV directory:", VOXCELEB_WAV_DIR)
print("Metadata directory:", METADATA_DIR)

Project root: /Users/juni/Documents/interview-anonymization
VoxCeleb WAV directory: /Users/juni/Documents/interview-anonymization/data/raw/voxceleb1/wav
Metadata directory: /Users/juni/Documents/interview-anonymization/data/metadata


## 2. Speaker and video inventories

In [2]:
speaker_df = voxceleb.build_speaker_inventory(VOXCELEB_WAV_DIR)

print("Speakers:", len(speaker_df))
print("Videos:", speaker_df["n_videos"].sum())
print("Utterances:", speaker_df["n_utterances"].sum())

display(speaker_df.describe())
display(speaker_df.sort_values("n_utterances", ascending=False).head(10))

Speakers: 1211
Videos: 21820
Utterances: 148642


,n_videos,n_utterances
count,1211.000000,1211.000000
mean,18.018167,122.743187
std,5.981763,77.902902
min,8.000000,45.000000
25%,13.000000,69.000000
50%,17.000000,102.000000
75%,22.000000,148.000000
max,36.000000,1002.000000


,speaker_id,n_videos,n_utterances
735,id10986,32,1002
183,id10096,16,656
974,id10020,27,589
291,id11111,23,570
647,id10710,26,489
1119,id10935,22,476
549,id11211,31,468
582,id10397,29,426
653,id10945,26,409
206,id10203,29,401


In [3]:
video_inventory_df = voxceleb.build_video_inventory(VOXCELEB_WAV_DIR)

if video_inventory_df.duplicated(["speaker_id", "video_id"]).any():
    raise ValueError("Duplicate speaker-video pairs found in the VoxCeleb inventory.")

print("Speaker-video pairs:", len(video_inventory_df))
display(video_inventory_df["n_utterances"].describe())

Speaker-video pairs: 21820


count    21820.000000
mean         6.812191
std          9.898486
min          1.000000
25%          2.000000
50%          4.000000
75%          8.000000
max        239.000000
Name: n_utterances, dtype: float64

## 3. Language metadata

In [4]:

from urllib.request import urlretrieve

LANGUAGE_CSV_URL = (
    "https://www.robots.ox.ac.uk/~vgg/data/voxceleb/"
    "data_workshop_2021/lang_vox1_final.csv"
)

LANGUAGE_CSV_PATH = METADATA_DIR / "lang_vox1_final.csv"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

if not LANGUAGE_CSV_PATH.is_file():
    print("Downloading language metadata...")
    urlretrieve(LANGUAGE_CSV_URL, LANGUAGE_CSV_PATH)

language_df = metadata.normalize_language_metadata(
    pd.read_csv(LANGUAGE_CSV_PATH)
)
language_video_df = metadata.aggregate_language_by_video(language_df)

print("Raw metadata shape:", language_df.shape)
print("Video-level metadata shape:", language_video_df.shape)
print("Missing language confidence:", language_df["language_confidence"].isna().sum())

display(language_df.sort_values("language_confidence").head(10))
display(language_df["language"].value_counts(dropna=False))

Raw metadata shape: (153516, 7)
Video-level metadata shape: (22496, 4)
Missing language confidence: 0


,filename,lang,confidence,speaker_id,video_id,language,language_confidence
94718,id10788/XQvOFXeDr2A/00001.wav,Haitian,0.63,id10788,XQvOFXeDr2A,haitian,0.63
148136,id11209/WYujrLqr28Y/00001.wav,Tatar,0.65,id11209,WYujrLqr28Y,tatar,0.65
148139,id11209/WYujrLqr28Y/00004.wav,Tatar,0.65,id11209,WYujrLqr28Y,tatar,0.65
148138,id11209/WYujrLqr28Y/00003.wav,Tatar,0.65,id11209,WYujrLqr28Y,tatar,0.65
148137,id11209/WYujrLqr28Y/00002.wav,Tatar,0.65,id11209,WYujrLqr28Y,tatar,0.65
92889,id10774/8NPeiO5hD7g/00002.wav,Maori,0.67,id10774,8NPeiO5hD7g,maori,0.67
92888,id10774/8NPeiO5hD7g/00001.wav,Maori,0.67,id10774,8NPeiO5hD7g,maori,0.67
76480,id10635/GHMhG_ARHXo/00002.wav,Bosnian,0.67,id10635,GHMhG_ARHXo,bosnian,0.67
76479,id10635/GHMhG_ARHXo/00001.wav,Bosnian,0.67,id10635,GHMhG_ARHXo,bosnian,0.67
114363,id10946/bHMtGYB65rE/00007.wav,Nepali,0.69,id10946,bHMtGYB65rE,nepali,0.69


language
english       129386
welsh          12944
hindi           1627
french          1611
german          1141
               ...  
ukrainian          1
macedonian         1
gujarati           1
latvian            1
hebrew             1
Name: count, Length: 84, dtype: int64

In [5]:
# Start from the raw video inventory each time, so this cell is safe to rerun.
video_df = video_inventory_df.merge(
    language_video_df,
    on=["speaker_id", "video_id"],
    how="left",
    validate="one_to_one",
)

print("Videos without language metadata:", video_df["language"].isna().sum())
display(video_df["language"].value_counts(dropna=False))

english_confidence = video_df.loc[
    video_df["language"].eq("english"),
    "language_confidence",
]
display(english_confidence.describe())

Videos without language metadata: 0


language
english      18486
welsh         1865
french         278
hindi          163
german         133
             ...  
shona            1
japanese         1
bulgarian        1
lao              1
gujarati         1
Name: count, Length: 84, dtype: int64

count    18486.000000
mean         0.963559
std          0.026638
min          0.750000
25%          0.960000
50%          0.970000
75%          0.980000
max          0.990000
Name: language_confidence, dtype: float64

## 4. Select the English-only subset

In [6]:
english_video_df = voxceleb.filter_eligible_english_videos(
    video_df,
    min_videos_per_speaker=3,
)

print("Final English-only subset")
print("Speakers:", english_video_df["speaker_id"].nunique())
print("Videos:", len(english_video_df))
print("Utterances:", english_video_df["n_utterances"].sum())

assert english_video_df["language"].eq("english").all()
assert (english_video_df.groupby("speaker_id").size() >= 3).all()

Final English-only subset
Speakers: 1166
Videos: 18449
Utterances: 124843


## 5. Create reproducible video-level splits

In [7]:
split_df = splits.create_speaker_video_splits(
    english_video_df,
    seed=42,
)

split_counts = split_df["split"].value_counts()
print("Videos per split:")
display(split_counts.to_frame("n_videos"))

print("Video proportions:")
display(split_df["split"].value_counts(normalize=True).to_frame("proportion"))

print("Speakers represented in each split:")
display(split_df.groupby("split")["speaker_id"].nunique().to_frame("n_speakers"))

speaker_split_counts = split_df.groupby("speaker_id")["split"].nunique()
print("Number of distinct splits per speaker:")
display(speaker_split_counts.value_counts().sort_index().to_frame("n_speakers"))

Videos per split:


,n_videos
split,
train,12834
val,2815
test,2800


Video proportions:


,proportion
split,
train,0.695647
val,0.152583
test,0.151770


Speakers represented in each split:


,n_speakers
split,
test,1166
train,1166
val,1166


Number of distinct splits per speaker:


,n_speakers
split,
3,1166


In [8]:
# Estimate utterance proportions from the per-video inventory.
split_utterance_counts = (
    split_df
    .merge(
        english_video_df[["speaker_id", "video_id", "n_utterances"]],
        on=["speaker_id", "video_id"],
        how="left",
        validate="one_to_one",
    )
    .groupby("split")["n_utterances"]
    .sum()
)

display(
    (split_utterance_counts / split_utterance_counts.sum())
    .rename("utterance_proportion")
    .to_frame()
)

,utterance_proportion
split,
test,0.147137
train,0.699655
val,0.153208


## 6. Build, validate, and save the utterance manifest

In [9]:
manifest_input_df = split_df.merge(
    english_video_df,
    on=["speaker_id", "video_id"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

if not manifest_input_df["_merge"].eq("both").all():
    raise ValueError("Some split videos have no matching video metadata.")

manifest_input_df = manifest_input_df.drop(columns="_merge")

utterance_df = voxceleb.build_utterance_manifest(
    manifest_input_df,
    VOXCELEB_WAV_DIR,
)

voxceleb.validate_utterance_manifest(
    utterance_df,
    voxceleb_wav_dir=VOXCELEB_WAV_DIR,
)

if not utterance_df["language"].eq("english").all():
    raise ValueError("The manifest contains non-English utterances.")

print("Manifest validation passed.")
print("Utterances:", len(utterance_df))
print("Speakers:", utterance_df["speaker_id"].nunique())
print("Videos:", utterance_df[["speaker_id", "video_id"]].drop_duplicates().shape[0])
print("Utterances per split:")
display(utterance_df["split"].value_counts().to_frame("n_utterances"))
display(utterance_df.head())

METADATA_DIR.mkdir(parents=True, exist_ok=True)
output_path = METADATA_DIR / "voxceleb_utterances.parquet"
utterance_df.to_parquet(output_path, index=False)

print("Saved:", output_path)

Manifest validation passed.
Utterances: 124843
Speakers: 1166
Videos: 18449
Utterances per split:


,n_utterances
split,
train,87347
val,19127
test,18369


,speaker_id,video_id,utterance_id,path,split,language,language_confidence
0,id10001,1zcIwhmdeo4,00001,id10001/1zcIwhmdeo4/00001.wav,train,english,0.99
1,id10001,1zcIwhmdeo4,00002,id10001/1zcIwhmdeo4/00002.wav,train,english,0.99
2,id10001,1zcIwhmdeo4,00003,id10001/1zcIwhmdeo4/00003.wav,train,english,0.99
3,id10001,7gWzIy6yIIk,00001,id10001/7gWzIy6yIIk/00001.wav,train,english,0.98
4,id10001,7gWzIy6yIIk,00002,id10001/7gWzIy6yIIk/00002.wav,train,english,0.98


Saved: /Users/juni/Documents/interview-anonymization/data/metadata/voxceleb_utterances.parquet
